# **This notebook acts as the data preprocessor of eICU**

In [ ]:
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
from sklearn.preprocessing import StandardScaler, OrdinalEncoder
from sklearn.feature_selection import VarianceThreshold
from sklearn.model_selection import train_test_split
import pyarrow.parquet as pq
import glob
import pyarrow as pa
import os 

## Loading our aggregated data

In [ ]:
def load_parquet_chunks_to_dataframe(file_pattern="parquets/client_2_part_*.parquet"):
    """
    Loads and concatenates parquet chunk files into a single pandas DataFrame.
    Args: file_pattern (str): The glob pattern to find the Parquet chunk files.
    Returns: pandas.DataFrame or None: The concatenated DataFrame if successful, otherwise None.
    """
    # Get a list of all your Parquet chunk files, sorted to maintain order
    parquet_files = sorted(glob.glob(file_pattern))

    if not parquet_files:
        print(f"No Parquet chunk files found matching pattern: '{file_pattern}'.")
        print("Please ensure the chunking process completed successfully and the files are in the correct directory.")
        return None
    else:
        print(f"Found {len(parquet_files)} Parquet files: {parquet_files}")

    # List to hold DataFrames from each chunk
    list_of_dataframes = []
    total_rows = 0

    # Read each Parquet chunk into a DataFrame
    for i, f_path in enumerate(parquet_files):
        try:
            if not os.path.exists(f_path):
                print(f"File not found: {f_path}. Skipping.")
                continue

            print(f"Reading chunk {i+1}/{len(parquet_files)}: {f_path}...")
            
            df_chunk = pd.read_parquet(f_path)
            list_of_dataframes.append(df_chunk)
            total_rows += len(df_chunk)

            print(f"Successfully read chunk {i+1}. Shape: {df_chunk.shape}. Rows so far: {total_rows}")

        except Exception as e:
            print(f"Error reading chunk {f_path}: {e}")
            # If one chunk fails, it's often better to stop and investigate
            # rather than concatenating partial data.
            print("Aborting concatenation due to error in reading a chunk.")
            return None

    if list_of_dataframes:
        # Concatenate all DataFrames in the list
        print("\nConcatenating all loaded chunks...")
        try:
            # Using ignore_index=True will create a new default integer index
            # for the combined DataFrame.
            combined_df = pd.concat(list_of_dataframes, ignore_index=True)
            print("Successfully concatenated all chunks.")
            print(f"Final DataFrame shape: {combined_df.shape}")
            print(f"Total rows in combined DataFrame: {len(combined_df)}")

            # Basic validation: check if total rows match
            if len(combined_df) == total_rows:
                print("Row count matches the sum of rows from individual chunks.")
            else:
                print(f"WARNING: Row count mismatch! Combined: {len(combined_df)}, Sum of chunks: {total_rows}")

            return combined_df
        except Exception as e:
            print(f"Error concatenating DataFrames: {e}")
            return None
    else:
        print("No DataFrames were read successfully, cannot concatenate.")
        return None

# How to use the function:
if __name__ == "__main__":
    # This pattern assumes your files are named like 'client_2_part_0.parquet', 'client_2_part_1.parquet', etc.
    # Adjust the pattern if your filenames are different.
    client_2_dataframe = load_parquet_chunks_to_dataframe(file_pattern="client_2_part_*.parquet")

    if client_2_dataframe is not None:
        print("\nSuccessfully loaded the dataset into 'client_2_dataframe'.")
        print("First 5 rows of the combined dataset:")
        print(client_2_dataframe.head())
        print("\nDataset info:")
        client_2_dataframe.info(memory_usage='deep')
    else:
        print("\nFailed to load the dataset.")

# **Skip the above cell and just run this, if the computer allows it:**

In [ ]:
client_2 = pd.read_csv('client_2_raw_hour.csv', low_memory=False)

In [ ]:
client_2 = client_2_dataframe

In [ ]:
client_2

## Understadig the structure of the dataset

## Checking the number of missing values

In [ ]:
missing_chunk = client_2.copy()

# Calculate missing value stats
missing_stats = (
    missing_chunk.isnull().sum()
    .to_frame(name='Missing_Count')
    .assign(Total=missing_chunk.shape[0])
    .assign(Missing_Percent=lambda x: (x['Missing_Count'] / x['Total']) * 100)
    .sort_values(by='Missing_Percent', ascending=False)
)

# Filter only columns with missing values
missing_stats_filtered = missing_stats[missing_stats['Missing_Count'] > 0]

missing_stats_filtered.shape[0] # Number of columns with missing values

In [ ]:
# Display top 20 columns with the highest percentage of missing values
top_missing = missing_stats_filtered.head(20).copy()
top_missing.reset_index(inplace=True)
top_missing.rename(columns={'index': 'Column Name'}, inplace=True)

top_missing

In [ ]:
missing_percent = client_2.isnull().mean().sort_values(ascending=False) * 100
missing_percent

In [ ]:
# Define a threshold for dropping columns based on missing percentage
drop_threshold = 95 # percent
columns_to_drop = missing_stats_filtered[missing_stats_filtered['Missing_Percent'] > drop_threshold].index.tolist()

# Separate remaining columns for potential imputation (not to be dropped)
columns_to_impute = missing_stats_filtered[
    (missing_stats_filtered['Missing_Percent'] <= drop_threshold)
].index.tolist()

# Show the first few columns that would be dropped and imputed
drop_and_impute_summary = {
    "Columns to drop (sample)": columns_to_drop[:10],
    "Drop count": len(columns_to_drop),
    "Columns to impute (sample)": columns_to_impute[:10],
    "Impute count": len(columns_to_impute)
}

drop_and_impute_summary

In [ ]:
missing_percent.head(30).plot(kind='bar', figsize=(12, 6), title="Top 30 features with most missing values")
plt.ylabel('% missing')
plt.tight_layout()
plt.ylim(0,100)
plt.show()

In [ ]:
client_2.drop(columns=columns_to_drop, inplace=True)

In [ ]:
client_2

### Like in MIMIC Extract handle cohorts

In [ ]:
print(client_2.columns.tolist())

In [ ]:
# To numeric:
client_2['age'] = client_2['age'].apply(pd.to_numeric, errors='coerce')

# AGAIN only keep adults, but since eICU has age, we can skip some steps and directly just do this:
adults = client_2[client_2['age'] >= 15].copy()

In [ ]:
adults['unitadmittime'] = pd.to_datetime(adults['unitadmittime'], errors='coerce')
adults['unitdischtime'] = pd.to_datetime(adults['unitdischtime'], errors='coerce')

# The 12 hours to 10 days stay filter:
first = (
    adults
    .sort_values(['patienthealthsystemstayid','unitadmittime'])
    .drop_duplicates(subset='patienthealthsystemstayid', keep='first')
    .copy()
)

first['los_hours'] = (
    first['unitdischtime'] - first['unitadmittime']
).dt.total_seconds() / 3600

cohort2 = first[
    (first['los_hours'] >= 12) &
    (first['los_hours'] < 240)
].copy()

In [ ]:
# Generated code for summary check just to be sure:
print("All stays:", first.shape[0])
print("Filtered cohort:", cohort2.shape[0])
print(cohort2['los_hours'].describe())

## Handling categorical variables

In [ ]:
cohort2['gender'].value_counts()

In [ ]:
cohort2_test = cohort2.copy()

cohort2_test['gender'] = (cohort2_test['gender'].map({'Male': 0, 'Female': 1}))

In [ ]:
cohort2_test['gender'].unique()

In [ ]:
# Locate the discharge-status column

status_cols = [c for c in cohort2_test.columns if 'dischargestatus' in c.lower()]

if not status_cols:
    raise KeyError("No discharge-status column found!")

status_col = status_cols[0]

print("Found status col:", status_col)
print(cohort2[status_col].value_counts())

In [ ]:
status_col = 'hospitaldischargestatus'
cohort2_test[status_col].str.lower().unique()

In [ ]:
status = cohort2_test[status_col].fillna('expired').str.title()

cohort2_test['is_alive'] = (status.map({'Alive': 1, 'Expired': 0}).astype(int))

cohort2_test['is_alive'].value_counts()

In [ ]:
cohort2_test.columns.tolist()

In [ ]:
cohort2_test['los_hours']

In [ ]:
cohort2_test

In [ ]:
# Now define exactly your static columns
# static_cols = ['subject_id','icustay_id','age','gender','los_hours','is_alive']
static_cols = ['patientunitstayid', 'age', 'gender', 'los_hours', 'is_alive', 'admissionheight', 'admissionweight', 'dischargeweight', ]
static = cohort2_test[static_cols].copy()
static

### Handling date time variables

In [ ]:
# From old preprocessing code:
static['admit_hour'] = cohort2['unitadmittime'].dt.hour

In [ ]:
# Cyclical (sin/cos) encodings so your model “knows” that 23 to 0h is adjacent:

# hour: 0 to 23 circle
static['hour_sin'] = np.sin(2*np.pi * static['admit_hour']   / 24)
static['hour_cos'] = np.cos(2*np.pi * static['admit_hour']   / 24)

In [ ]:
static

In [ ]:
static.dtypes

### Putting it all together

In [ ]:
dyn_cols = sorted(set(cohort2_test.columns) - set(static_cols))
X_dyn = cohort2_test[dyn_cols].copy()

In [ ]:
print(dyn_cols)

In [ ]:
import re

# 1) select only columns named like "<var>_h<hour>"
pat = re.compile(r'^(?P<var>.+)_h(?P<hour>\d+)$')
valid = [c for c in X_dyn.columns if pat.match(c)]

hours = sorted({int(pat.match(c).group('hour')) for c in valid})
vars_ = sorted({pat.match(c).group('var') for c in valid})

complete_vars = [
    v for v in vars_
    if all(f"{v}_h{h}" in X_dyn.columns for h in hours)
]

ordered = [
    f"{v}_h{h}"
    for h in hours
    for v in complete_vars
]

In [ ]:
to_stack = X_dyn[ordered]

# Which dtypes do we actually have?
print(to_stack.dtypes.value_counts())

# List the object-dtype (string) columns
bad = to_stack.dtypes[to_stack.dtypes == "object"].index.tolist()
print("non-numeric dynamic cols:", bad)

In [ ]:
# Filter to only true numeric columns
numeric_ordered = [c for c in ordered
                   if pd.api.types.is_numeric_dtype(X_dyn[c])]

# Re-compute 3-D array out of those:
arr = X_dyn[numeric_ordered].values.astype("float32")
N, HM = arr.shape
H     = len(hours)
M     = HM // H

X_3d = arr.reshape(N, H, M)
print("Now dynamic shape:", X_3d.shape, "dtype=", X_3d.dtype)

# **TRAIN TEST SPLIT, IMPUTING AND NORMALIZATION**

In [ ]:
Xs_num = static.drop(columns=["patientunitstayid", "is_alive"])

In [ ]:
non_numeric = [c for c in Xs_num.columns
               if not pd.api.types.is_numeric_dtype(Xs_num[c])]
if non_numeric:
    raise ValueError("Non-numeric column(s) in static:", non_numeric)

In [ ]:
Xs = Xs_num.values.astype("float32") 
y = static["is_alive"].values.astype("float32")

In [ ]:
from sklearn.model_selection import train_test_split

idx = np.arange(N)
tr_idx, te_idx = train_test_split(idx, test_size=0.2, stratify=y, random_state=42)

# Xs for static features, Xd for dynamic features

Xs_tr, Xs_te = Xs[tr_idx], Xs[te_idx]
Xd_tr, Xd_te = X_3d[tr_idx], X_3d[te_idx]
y_tr, y_te = y[tr_idx], y[te_idx]

In [ ]:
print("train:", Xs_tr.shape, Xd_tr.shape, y_tr.shape)
print("test: ", Xs_te.shape, Xd_te.shape, y_te.shape)

In [ ]:
from sklearn.impute import SimpleImputer
from sklearn.preprocessing import StandardScaler

# Static pipeline
imp_s = SimpleImputer(strategy="mean").fit(Xs_tr) # Imputer
sc_s = StandardScaler().fit(imp_s.transform(Xs_tr)) # StandardScaler

Xs_tr = sc_s.transform(imp_s.transform(Xs_tr))
Xs_te = sc_s.transform(imp_s.transform(Xs_te))

# Dynamic pipeline: reshape to (N, H*M), impute/scale, then back to (N,H,M)
Xd_tr_flat = Xd_tr.reshape(len(Xd_tr), -1)
Xd_te_flat = Xd_te.reshape(len(Xd_te), -1)

imp_d = SimpleImputer(strategy="mean").fit(Xd_tr_flat) # Imputer
sc_d = StandardScaler().fit(imp_d.transform(Xd_tr_flat))

Xd_tr = sc_d.transform(imp_d.transform(Xd_tr_flat)).reshape(-1, H, M)
Xd_te = sc_d.transform(imp_d.transform(Xd_te_flat)).reshape(-1, H, M)

In [ ]:
print("static NaNs in train:", np.isnan(Xs_tr).sum())
print("static NaNs in  test:", np.isnan(Xs_te).sum())
print("dynamic NaNs in train:", np.isnan(Xd_tr).sum())
print("dynamic NaNs in  test:", np.isnan(Xd_te).sum())

# **Saving to NUMPY arrays**

In [ ]:
np.save("Xstatic_train_c2.npy", Xs_tr)
np.save("Xstatic_test_c2.npy", Xs_te)
np.save("Xdynamic_train_c2.npy", Xd_tr)
np.save("Xdynamic_test_c2.npy", Xd_te)
np.save("y_train_c2.npy", y_tr)
np.save("y_test_c2.npy",  y_te)

In [ ]:
static.to_csv("client_2_static.csv", index=False)

In [ ]:
client_2_dynamic = cohort2_test[dyn_cols].copy()

client_2_dynamic.to_csv("client_2_dynamic.csv", index=False)

# **FOR FL for matching static features**

In [ ]:
Xs_matching = static.drop(columns=["patientunitstayid","admissionheight", "admissionweight", "dischargeweight", "is_alive"]).copy()

In [ ]:
Xs_local = static.drop(columns=["patientunitstayid", "age", "gender", "los_hours", "hour_sin", "hour_cos", "is_alive", "admit_hour"]).copy()

In [ ]:
Xs_matching

In [ ]:
Xs_local

In [ ]:
non_numeric = [c for c in Xs_local.columns
               if not pd.api.types.is_numeric_dtype(Xs_local[c])]
if non_numeric:
    raise ValueError("Non-numeric column(s) in static:", non_numeric)

Xs_local = Xs_local.values.astype("float32") 
y = static["is_alive"].values.astype("float32")

from sklearn.model_selection import train_test_split

idx = np.arange(N)
tr_idx, te_idx = train_test_split(idx, test_size=0.2, stratify=y, random_state=42)

Xs_tr_local, Xs_te_local= Xs_local[tr_idx], Xs_local[te_idx]
y_tr_local, y_te_local = y[tr_idx], y[te_idx]

In [ ]:
print("train:", Xs_tr_local.shape)
print("test: ", Xs_te_local.shape)

In [ ]:
from sklearn.impute import SimpleImputer
from sklearn.preprocessing import StandardScaler

# # # Static pipeline
# imp_s = SimpleImputer(strategy="mean").fit(Xs_tr_match) # Imputer
# sc_s = StandardScaler().fit(imp_s.transform(Xs_tr_match)) # StandardScaler

# Xs_tr_match = sc_s.transform(imp_s.transform(Xs_tr_match))
# Xs_te = sc_s.transform(imp_s.transform(Xs_te_match))

# 1) First pass: mean‐impute whatever you can
imp_mean = SimpleImputer(strategy="mean").fit(Xs_tr_local)
Xs_tr_imp = imp_mean.transform(Xs_tr_local)
Xs_te_imp = imp_mean.transform(Xs_te_local)

# 2) Second pass: fill any remaining NaNs (columns that were all-NaN in train) with zero
imp_zero  = SimpleImputer(strategy="constant", fill_value=0.0)
Xs_tr_imp = imp_zero.fit_transform(Xs_tr_imp)
Xs_te_imp = imp_zero.transform(Xs_te_imp)

# 3) Finally scale
sc_s = StandardScaler().fit(Xs_tr_imp)
Xs_tr_scaled = sc_s.transform(Xs_tr_imp)
Xs_te_scaled = sc_s.transform(Xs_te_imp)

Xs_tr_local = Xs_tr_scaled
Xs_te_local = Xs_te_scaled

In [ ]:
print("static NaNs in train:", np.isnan(Xs_tr_local).sum())
print("static NaNs in  test:", np.isnan(Xs_te_local).sum())

In [ ]:
np.save("Xstatic_train_local2.npy", Xs_tr_local)
np.save("Xstatic_test_local2.npy", Xs_te_local)

In [ ]:
# np.save("y_train_match2.npy", y_tr_match)
# np.save("y_test_match2.npy",  y_te_match)